In [12]:
import torch 
import torch.nn.functional as F 
import matplotlib.pyplot as plt 

In [13]:
g = torch.Generator().manual_seed(2147483647)

In [15]:
words = open("names.txt", 'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [16]:
chars = sorted(list(set("".join(words))))

stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0

itos = {i:s for s, i in stoi.items()}

In [17]:
# Build dataset function with train, dev and test splits 
block_size = 3 
def build_dataset(words):
    X, Y = [], [] 

    for w in words: 
        context = [0] * block_size 

        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random 
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [8]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn(fan_in, fan_out, generator=g) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None 
    
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out 
    
    def parameters(self):
        return [self.weight] + [[] if self.bias is None else self.bias]


class BatchNorm:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        # constants 
        self.eps = eps
        self.momentum = momentum
        self.training = True 
        # trainable parameters 
        self.gamma = torch.ones(dim) # scale
        self.beta = torch.zeros(dim) # shift
        # running mean and var
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        # forward pass 
        if self.training: 
            xmean = x.mean(0, keepdim=True) # batch mean 
            xvar = x.var(0, keepdim=True) # batch variance 
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # standardize normalization 
        self.out = self.gamma * xhat + self.beta # scale and shift 
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean 
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar 

        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self): 
        return [] # tanh has no parameters 

In [11]:
n_embd = 10 # dimensionality of the character embedding vectors 
n_hidden = 100 # number of neurons at the hidden layers of the mlp 
vocab_size = 27
block_size = 3

C = torch.randn(vocab_size, n_embd, generator=g)

layers = [
    Linear(block_size * n_embd, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, n_hidden), Tanh(),
    Linear(             n_hidden, vocab_size),
]


# DOn't keep track of compututional graph of these init operations - hence require no grad
with torch.no_grad(): # we 
    # scale down the weights of the last layer so that we have a uniform weight dist
    layers[-1].weight *= 0.1 # it prevents overconfident predictions leading to a very high loss in eary training steps 
    for layer in layers[:-1]: # for the remaining layers 
        if isinstance(layer, Linear): # if the layer is a linear layer 
            layer.weight *= 5/3 # use the kaiming init - the divison by fan_in**0.5 has been added to layer.weight in Linear class


parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # total number of parameter 

for p in parameters:
    p.requires_grad = True 

46497


In [18]:
Xtr.shape

torch.Size([182625, 3])

In [20]:
ix = torch.randint(0, Xtr.shape[0], (32,))
ix

tensor([ 26061,  52221, 141172,  88575,  48067, 176985,  76472, 141751, 181499,
        109881,   7809, 172335,  94360,  84680, 176063, 120360,  64057,  26138,
        151609, 133853, 158990,  58134, 158347,  18063,  18752, 120178, 139864,
        131808,  31308,  97254,   1749, 148325])

In [ ]:
# Training loop 

max_steps = 200_000
batch_size = 32 
lossi = [] 

for i in range(max_steps):

    # mini batch construct 
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)

    # forward pass 
    emb = C[Xtr[ix]]
    emb_cat = emb.view(emb.shape[0], -1)
    for layer in layers: 
        x = layer(x)
    loss = F.cross_entropy(x, Ytr[ix]) # loss function 

    # backward pass 
    for layer in layers: 
        layer.out.retain_grad()
    for p in parameters:
        p.grad = None 
    loss.backward()

    # update 
    lr = 0.1 if max_steps < 10_000 else 0.1 # applied learning rate decay 
    for p in parameters:
        p.data -= lr * p.grad

    # track stats 
    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())